# Pengambilan dan Pembersihan Data

Proyek: Analisis Sentimen, Clustering, dan Regresi Likes pada Postingan X tentang Timnas

Penanggung jawab: Muhammad Yafi Adyatma, 24051130006

Notebook ini menghasilkan `dataset_timnas.csv` dengan kolom `text`, `created_at`, dan `like_count`, ditambah berkas dokumentasi yang dibaca oleh aplikasi Streamlit. Jalankan notebook ini dari folder `notebooks/` karena semua path bersifat relatif.

## 1. Dokumentasi Sumber Data

| Item | Isi |
|---|---|
| Platform | X, sebelumnya Twitter |
| Topik | Sepak bola dan Tim Nasional Indonesia |
| Kata kunci | timnas indonesia, timnas, garuda, #TimnasDay, pssi |
| Bahasa | Indonesia, memakai filter `lang:id` |
| Metode | Crawling memakai tweet-harvest melalui notebook ini, atau dataset cadangan yang disetujui dosen |
| Keluaran | `data/processed/dataset_timnas.csv`, `data/processed/metadata_pengambilan.json`, `data/processed/cleaning_log.csv` |

Syarat menjalankan crawling:

1. Node.js versi 18 atau lebih baru sudah terpasang.
2. Memiliki auth token dari cookie akun X yang sedang login.
3. Token disimpan pada environment variable `X_AUTH_TOKEN` dan tidak boleh ditulis di notebook atau diunggah ke repositori.

Jika crawling tidak bisa dilakukan, letakkan dataset cadangan berformat CSV di `data/raw/dataset_timnas_mentah.csv`, lalu ubah `RUN_CRAWL` menjadi `False` pada sel pengambilan data.

## 2. Konfigurasi

In [ ]:
import os
import re
import glob
import json
import shutil
import subprocess
from datetime import datetime

import pandas as pd

KEYWORDS = ["timnas indonesia", "timnas", "garuda", "#TimnasDay", "pssi"]
SINCE = "2026-10-01"
UNTIL = "2026-10-09"
LIMIT_PER_KEYWORD = 150
MIN_ROWS = 150

AUTH_TOKEN = os.getenv("X_AUTH_TOKEN", "")
# Path relatif terhadap folder notebooks/
RAW_DIR = os.path.join("..", "data", "raw")
CRAWL_DIR = os.path.join(RAW_DIR, "crawl")
RAW_CSV = os.path.join(RAW_DIR, "dataset_timnas_mentah.csv")
PROCESSED_DIR = os.path.join("..", "data", "processed")
OUTPUT_CSV = os.path.join(PROCESSED_DIR, "dataset_timnas.csv")

os.makedirs(CRAWL_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 3. Pengambilan Data

Setiap kata kunci diambil dalam berkas CSV terpisah di folder `data/raw/crawl/`. Gabungan semuanya disimpan sebagai `data/raw/dataset_timnas_mentah.csv`. Kegagalan pada satu kata kunci dicatat sebagai kendala dan tidak menghentikan kata kunci lainnya.

In [ ]:
RUN_CRAWL = True
METODE = "Crawling dengan tweet-harvest melalui notebook"

kendala = []
WAKTU_PENGAMBILAN = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def slug(teks):
    return re.sub(r"[^a-z0-9]+", "_", teks.lower()).strip("_")

if RUN_CRAWL:
    npx = shutil.which("npx")
    if npx is None:
        raise RuntimeError("Node.js belum terpasang, npx tidak ditemukan.")
    if not AUTH_TOKEN:
        raise RuntimeError("Isi environment variable X_AUTH_TOKEN sebelum crawling.")

    for kw in KEYWORDS:
        nama = f"{slug(kw)}.csv"
        query = f"{kw} lang:id since:{SINCE} until:{UNTIL}"
        hasil = subprocess.run(
            [npx, "-y", "tweet-harvest@latest", "-o", nama, "-s", query,
             "--tab", "LATEST", "-l", str(LIMIT_PER_KEYWORD), "--token", AUTH_TOKEN],
            capture_output=True, text=True,
        )
        sumber = os.path.join("tweets-data", nama)
        if os.path.exists(sumber):
            shutil.move(sumber, os.path.join(CRAWL_DIR, nama))
            print(f"Berhasil: {kw}")
        else:
            kendala.append(f"Crawling gagal untuk kata kunci {kw}")
            print(f"Gagal: {kw}")
            print(hasil.stderr[-400:])
else:
    METODE = "Dataset cadangan yang disetujui dosen"

## 4. Memuat dan Menyeragamkan Kolom

Nama kolom hasil crawling berbeda antar alat, sehingga dipetakan ke nama kolom standar proyek.

In [ ]:
frames = []
for path in sorted(glob.glob(os.path.join(CRAWL_DIR, "*.csv"))):
    bagian = pd.read_csv(path)
    bagian["kata_kunci"] = os.path.splitext(os.path.basename(path))[0]
    frames.append(bagian)

if frames:
    raw = pd.concat(frames, ignore_index=True)
    raw.to_csv(RAW_CSV, index=False, encoding="utf-8")
elif os.path.exists(RAW_CSV):
    # Dataset cadangan yang diletakkan langsung di data/raw
    raw = pd.read_csv(RAW_CSV)
else:
    raise FileNotFoundError("Tidak ada data mentah di data/raw.")

if "kata_kunci" not in raw.columns:
    raw["kata_kunci"] = "dataset_cadangan"

KANDIDAT = {
    "text": ["full_text", "text", "content"],
    "created_at": ["created_at", "date", "tweet_created_at"],
    "like_count": ["favorite_count", "like_count", "likes", "likeCount"],
    "repost_count": ["retweet_count", "repost_count", "retweets"],
    "reply_count": ["reply_count", "replies"],
    "url": ["tweet_url", "url"],
}

def ambil_kolom(df, kandidat):
    for nama in kandidat:
        if nama in df.columns:
            return df[nama]
    return pd.Series([None] * len(df), index=df.index)

df = pd.DataFrame({k: ambil_kolom(raw, v) for k, v in KANDIDAT.items()})
df["kata_kunci"] = raw["kata_kunci"]

if df["text"].isna().all():
    raise ValueError("Kolom teks tidak ditemukan pada data mentah.")

print("Jumlah data mentah:", len(df))
df.head()

## 5. Pembersihan Data

Urutan tahap: teks kosong, duplikat, kesesuaian topik, validasi `like_count`, dan pembuatan `text_clean`. Jumlah baris setiap tahap dicatat untuk laporan.

In [ ]:
log = []

def catat(tahap, data):
    log.append({"tahap": tahap, "jumlah_baris": len(data)})

catat("Data mentah", df)

# Teks kosong dan nilai hilang
df = df.dropna(subset=["text"])
df = df[df["text"].astype(str).str.strip() != ""]
catat("Hapus teks kosong", df)

# Duplikat berdasarkan teks asli
df = df.drop_duplicates(subset="text")
catat("Hapus duplikat teks asli", df)

In [ ]:
# Kesesuaian topik. Kata garuda saja bisa merujuk ke maskapai, sehingga dikecualikan.
teks_kecil = df["text"].astype(str).str.lower()
inti = teks_kecil.str.contains(r"timnas|tim nasional|pssi|sepak ?bola")
garuda = teks_kecil.str.contains("garuda") & ~teks_kecil.str.contains(
    r"garuda indonesia|maskapai|penerbangan|flight"
)
df = df[inti | garuda]
catat("Buang postingan tidak relevan", df)

In [ ]:
# Negasi seperti tidak dan bukan sengaja dipertahankan karena memengaruhi sentimen
def bersihkan_teks(teks):
    teks = str(teks).lower()
    teks = re.sub(r"^rt\s+", "", teks)
    teks = re.sub(r"http\S+|www\.\S+", " ", teks)
    teks = re.sub(r"@\w+", " ", teks)
    teks = re.sub(r"[^a-z0-9\s]", " ", teks)
    return re.sub(r"\s+", " ", teks).strip()

df["text_clean"] = df["text"].apply(bersihkan_teks)
df = df[df["text_clean"] != ""]
catat("Hapus teks bersih kosong", df)

df = df.drop_duplicates(subset="text_clean")
catat("Hapus duplikat teks bersih", df)

In [ ]:
# Validasi like_count. Nilai tidak valid menjadi kosong dan tidak diganti dengan nol
like = pd.to_numeric(df["like_count"], errors="coerce")
like = like.where(like >= 0)
df["like_count"] = like.round().astype("Int64")

for kolom in ["repost_count", "reply_count"]:
    df[kolom] = pd.to_numeric(df[kolom], errors="coerce").round().astype("Int64")

# Waktu publikasi disimpan dalam WIB
waktu = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
df["created_at"] = waktu.dt.tz_convert("Asia/Jakarta").dt.strftime("%Y-%m-%d %H:%M:%S")

jumlah_like_kosong = int(df["like_count"].isna().sum())
jumlah_waktu_kosong = int(df["created_at"].isna().sum())
catat("Dataset akhir", df)

print("Like tidak valid:", jumlah_like_kosong)
print("Waktu tidak valid:", jumlah_waktu_kosong)
pd.DataFrame(log)

## 6. Menyimpan Dataset dan Dokumentasi

In [ ]:
kolom_akhir = ["text", "created_at", "like_count", "repost_count",
               "reply_count", "url", "kata_kunci", "text_clean"]
df = df[kolom_akhir].reset_index(drop=True)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

pd.DataFrame(log).to_csv(os.path.join(PROCESSED_DIR, "cleaning_log.csv"), index=False)

if len(df) < MIN_ROWS:
    kendala.append(
        f"Jumlah data akhir {len(df)} kurang dari target minimal {MIN_ROWS} postingan unik"
    )
if jumlah_like_kosong > 0:
    kendala.append(f"{jumlah_like_kosong} postingan memiliki like_count yang tidak valid")

# Isi catatan tambahan sendiri bila ada kendala lain selama pengambilan data
KENDALA_MANUAL = []

metadata = {
    "topik": "Sepak bola dan Tim Nasional Indonesia",
    "kata_kunci": KEYWORDS,
    "periode_mulai": SINCE,
    "periode_selesai": UNTIL,
    "metode": METODE,
    "waktu_pengambilan": WAKTU_PENGAMBILAN,
    "jumlah_data_mentah": int(log[0]["jumlah_baris"]),
    "jumlah_data_akhir": int(len(df)),
    "kendala": kendala + KENDALA_MANUAL,
}
with open(os.path.join(PROCESSED_DIR, "metadata_pengambilan.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Dataset tersimpan: {OUTPUT_CSV}, {len(df)} baris")
df.head()

## 7. Ringkasan Hasil

In [ ]:
print("Jumlah postingan unik:", len(df))
print("Target minimal tercapai:", len(df) >= MIN_ROWS)
print("Rentang waktu:", df["created_at"].min(), "sampai", df["created_at"].max())
print()
print(df["like_count"].describe())
print()
print(df["kata_kunci"].value_counts())

## 8. Dokumentasi Kendala

Kendala yang tercatat otomatis tersimpan pada `metadata_pengambilan.json` dan ditampilkan di tab Data. Kendala yang umum muncul pada tahap ini:

- Crawling membutuhkan auth token akun X yang bisa kedaluwarsa atau dibatasi saat dipakai berulang.
- Jumlah hasil per kata kunci dibatasi oleh `LIMIT_PER_KEYWORD` dan oleh batas pemuatan halaman pencarian X.
- Kata kunci yang luas seperti garuda memunculkan postingan tidak relevan, sehingga perlu filter topik.
- Banyak postingan berupa retweet atau salinan yang menyebabkan duplikat.
- Nilai `like_count` adalah jumlah saat data diambil, sehingga postingan baru cenderung memiliki likes lebih sedikit.

Tambahkan kendala lain yang benar-benar dialami pada daftar `KENDALA_MANUAL` di sel penyimpanan, lalu jalankan ulang sel tersebut.